# 🎮 Minecraft for Colab (AIサバイバルサーバー)

一時的なMinecraft Java版サーバーをGoogle Colab上で起動し、**追加費用0円でLLMを搭載した4体のAI仲間（Gemini, ChatGPT, Claude, Grok）** と日本語チャットしながらサバイバル生活を楽しむための自動化ノートブックです。

### 🌟 特徴
- **完全0円のLLM**: L4 GPU (24GB VRAM) を活用したローカル推論 (Qwen2.5/Ollama) または Google AI Studio (Gemini 1.5 Free Tier)。
- **4人の個性的なAIPlayer**: Gemini（建築・進行）、ChatGPT（採掘・素材管理）、Claude（護衛・安全）、Grok（探索・冒険）が同時に参加。
- **自己完結型ワンクリック起動**: 認証不要で1セル実行するだけで、Google Driveマウント・全スクリプト展開・サーバー構築・playit.ggトンネル・Bot参加・録画開始まで全自動で完了。
- **データ消失防止**: Google Drive および XServer へのワールドデータ＆録画動画の自動同期・復元。
- **自動動画収録**: Bot視点を3Dレンダリングし、プレイ動画（MP4）を自動録画保存。

---
## 🚀 ステップ 1: 【ワンクリック起動】
**下のセルを実行（再生ボタンを押す）するだけ** で、すべての準備・起動・Bot参加・録画が全自動で始まります。  
画面に表示される **`playit.gg` の接続アドレス** を手元のMinecraft（Java版）に入力してログインしてください。

In [ ]:
#@title 🎮 ワンクリック起動（サーバー・0円LLM・4体Bot・録画）
import os
import sys
import json
import subprocess
import time

print("==================================================")
print("  Minecraft AI Colab Server - 初期化中...")
print("==================================================")

# 1. Google Drive の自動マウント（データ永続化用）
try:
    from google.colab import drive
    if not os.path.exists('/content/drive/MyDrive'):
        print('📂 Google Drive をマウント中...')
        drive.mount('/content/drive')
except Exception as e:
    print('⚠️ Google Driveのマウントをスキップ (ローカル保存で継続):', e)

# 2. リポジトリの準備（自動展開）
REPO_PATH = '/content/ColabTool'
os.makedirs(REPO_PATH, exist_ok=True)

# 常に最新スクリプトを展開・配置
EMBEDDED_FILES = {"scripts/setup_server.sh": "#!/bin/bash\n# ==============================================================================\n# setup_server.sh - Minecraft Paper/Purpur Server & playit.gg Auto Setup\n# ==============================================================================\nset -e\n\nSERVER_DIR=\"/content/minecraft_server\"\nMINECRAFT_VERSION=\"1.21.1\" # 最新安定版\nPLAYIT_VERSION=\"0.15.26\"\n\necho \"=== [1/4] Java 21 および必要パッケージのセットアップ ===\"\napt-get update -qq\napt-get install -y -qq openjdk-21-jre-headless curl jq screen xvfb ffmpeg x11-utils > /dev/null 2>&1\necho \"Javaバージョン:\"\njava -version 2>&1 | head -n 1\n\necho \"=== [2/4] Minecraft サーバーディレクトリ作成 ===\"\nmkdir -p \"$SERVER_DIR\"\ncd \"$SERVER_DIR\"\n\n# EULA 同意\necho \"eula=true\" > eula.txt\n\n# server.properties 最適化\ncat << 'EOF' > server.properties\nserver-port=25565\nonline-mode=false\ndifficulty=normal\ngamemode=survival\npvp=true\nview-distance=8\nsimulation-distance=6\nmax-players=10\nenable-command-block=true\nmotd=Minecraft AI Colab Server\nnetwork-compression-threshold=256\nsync-chunk-writes=false\nallow-flight=true\nspawn-protection=0\nEOF\n\necho \"=== [3/4] PaperMC (Java 21 最適化サーバー) のダウンロード ===\"\nif [ ! -f \"server.jar\" ]; then\n    # PaperMC APIから最新ビルドを取得\n    PAPER_PROJECT=\"paper\"\n    LATEST_BUILD=$(curl -s \"https://api.papermc.io/v2/projects/${PAPER_PROJECT}/versions/${MINECRAFT_VERSION}\" | jq '.builds[-1]')\n    if [ \"$LATEST_BUILD\" != \"null\" ] && [ -n \"$LATEST_BUILD\" ]; then\n        JAR_NAME=\"${PAPER_PROJECT}-${MINECRAFT_VERSION}-${LATEST_BUILD}.jar\"\n        DOWNLOAD_URL=\"https://api.papermc.io/v2/projects/${PAPER_PROJECT}/versions/${MINECRAFT_VERSION}/builds/${LATEST_BUILD}/downloads/${JAR_NAME}\"\n        echo \"PaperMC Build #${LATEST_BUILD} をダウンロード中: $DOWNLOAD_URL\"\n        curl -s -o server.jar \"$DOWNLOAD_URL\"\n    else\n        echo \"Purpurのフォールバックダウンロード...\"\n        curl -s -o server.jar \"https://api.purpurmc.org/v2/purpur/${MINECRAFT_VERSION}/latest/download\"\n    fi\n    echo \"server.jar の準備完了\"\nelse\n    echo \"既存の server.jar が見つかりました。\"\nfi\n\necho \"=== [4/4] playit.gg (無料トンネルCLI) のダウンロード ===\"\nif [ ! -f \"/usr/local/bin/playit\" ]; then\n    echo \"playit.gg CLI をインストール中...\"\n    curl -SsL -o /usr/local/bin/playit \"https://github.com/playit-cloud/playit-agent/releases/download/v${PLAYIT_VERSION}/playit-linux-amd64\"\n    chmod +x /usr/local/bin/playit\nfi\necho \"playit.gg の準備完了\"\n\necho \"=== サーバーセットアップ完了 ===\"\n", "scripts/backup_restore.py": "#!/usr/bin/env python3\n\"\"\"\nbackup_restore.py - Minecraft Server Data & Recording Sync (Google Drive & XServer)\n\"\"\"\n\nimport os\nimport sys\nimport shutil\nimport tarfile\nimport subprocess\nfrom datetime import datetime\nfrom pathlib import Path\n\nSERVER_DIR = Path(\"/content/minecraft_server\")\nLOCAL_BACKUP_DIR = Path(\"/content/backup_temp\")\nGDRIVE_DIR = Path(\"/content/drive/MyDrive/Minecraft_Colab\")\nRECORDINGS_DIR = Path(\"/content/recordings\")\n\n# 保存対象のディレクトリ・ファイル\nSAVE_TARGETS = [\n    \"world\",\n    \"world_nether\",\n    \"world_the_end\",\n    \"server.properties\",\n    \"whitelist.json\",\n    \"ops.json\",\n    \"banned-players.json\",\n    \"plugins\"\n]\n\ndef log(msg: str):\n    print(f\"[{datetime.now().strftime('%Y-%m-%d %H:%M:%S')}] {msg}\", flush=True)\n\ndef restore_from_gdrive():\n    \"\"\"Google Driveから最新のワールドデータを復元\"\"\"\n    if not GDRIVE_DIR.exists():\n        log(\"Google Driveがマウントされていないか、Minecraft_Colabフォルダが見つかりません。新規ワールドで開始します。\")\n        return False\n\n    latest_archive = None\n    archives = sorted(GDRIVE_DIR.glob(\"mc_world_*.tar.gz\"), key=os.path.getmtime, reverse=True)\n    if archives:\n        latest_archive = archives[0]\n        log(f\"Google Driveから最新アーカイブを検出: {latest_archive.name}\")\n        SERVER_DIR.mkdir(parents=True, exist_ok=True)\n        with tarfile.open(latest_archive, \"r:gz\") as tar:\n            tar.extractall(path=SERVER_DIR)\n        log(\"ワールドデータの復元が完了しました。\")\n        return True\n    else:\n        log(\"Google Drive内に既存のワールドアーカイブが見つかりません。新規作成します。\")\n        return False\n\ndef backup_to_gdrive():\n    \"\"\"Google Driveへワールドデータを保存\"\"\"\n    if not SERVER_DIR.exists():\n        log(\"サーバーディレクトリが存在しません。\")\n        return False\n\n    GDRIVE_DIR.mkdir(parents=True, exist_ok=True)\n    timestamp = datetime.now().strftime(\"%Y%m%d_%H%M%S\")\n    archive_name = f\"mc_world_{timestamp}.tar.gz\"\n    archive_path = GDRIVE_DIR / archive_name\n    latest_path = GDRIVE_DIR / \"mc_world_latest.tar.gz\"\n\n    log(f\"ワールドデータを圧縮中: {archive_path.name}\")\n    with tarfile.open(archive_path, \"w:gz\") as tar:\n        for target in SAVE_TARGETS:\n            target_path = SERVER_DIR / target\n            if target_path.exists():\n                tar.add(target_path, arcname=target)\n\n    # latestとしてもコピー\n    shutil.copy2(archive_path, latest_path)\n    log(\"Google Driveへのバックアップが完了しました。\")\n\n    # 録画動画の同期\n    if RECORDINGS_DIR.exists():\n        gdrive_rec_dir = GDRIVE_DIR / \"recordings\"\n        gdrive_rec_dir.mkdir(parents=True, exist_ok=True)\n        for mp4 in RECORDINGS_DIR.glob(\"*.mp4\"):\n            dest = gdrive_rec_dir / mp4.name\n            if not dest.exists() or dest.stat().st_size != mp4.stat().st_size:\n                log(f\"録画動画をGoogle Driveに同期中: {mp4.name}\")\n                shutil.copy2(mp4, dest)\n    return True\n\ndef sync_to_xserver(host, user, port, key_path, remote_dir):\n    \"\"\"XServerへrsyncで同期（オプション）\"\"\"\n    if not host or not user:\n        return\n    log(f\"XServer ({host}) へのバックアップ同期を開始...\")\n    archive_latest = GDRIVE_DIR / \"mc_world_latest.tar.gz\"\n    if not archive_latest.exists():\n        log(\"送信するアーカイブがありません。\")\n        return\n\n    ssh_cmd = f\"ssh -p {port} -i {key_path} -o StrictHostKeyChecking=no\"\n    cmd = [\n        \"rsync\", \"-avz\", \"-e\", ssh_cmd,\n        str(archive_latest),\n        f\"{user}@{host}:{remote_dir}/\"\n    ]\n    try:\n        subprocess.run(cmd, check=True)\n        log(\"XServerへの同期が成功しました。\")\n    except Exception as e:\n        log(f\"XServer同期エラー: {e}\")\n\nif __name__ == \"__main__\":\n    action = sys.argv[1] if len(sys.argv) > 1 else \"backup\"\n    if action == \"restore\":\n        restore_from_gdrive()\n    elif action == \"backup\":\n        backup_to_gdrive()\n        # XServer環境変数があれば同期\n        x_host = os.environ.get(\"XSERVER_HOST\")\n        x_user = os.environ.get(\"XSERVER_USER\")\n        x_port = os.environ.get(\"XSERVER_PORT\", \"22\")\n        x_key = os.environ.get(\"XSERVER_KEY\", \"/root/.ssh/id_rsa\")\n        x_dir = os.environ.get(\"XSERVER_REMOTE_DIR\", \"~/minecraft_backup\")\n        if x_host and x_user:\n            sync_to_xserver(x_host, x_user, x_port, x_key, x_dir)\n    else:\n        print(f\"Unknown action: {action}\")\n", "scripts/start_all.sh": "#!/bin/bash\n# ==============================================================================\n# start_all.sh - ワンクリック完全起動スクリプト (Colab堅牢化版)\n# (サーバー・トンネル・LLM・Bot・録画・バックアップ・動向監視)\n# ==============================================================================\n\nREPO_DIR=\"$(cd \"$(dirname \"${BASH_SOURCE[0]}\")/..\" && pwd)\"\nSERVER_DIR=\"/content/minecraft_server\"\nmkdir -p \"$SERVER_DIR\"\n\necho \"==================================================\"\necho \"    Minecraft AI Colab Server - 自動起動開始\"\necho \"==================================================\"\n\n# 1. サーバー環境 & ツールセットアップ\necho \">> [1/7] システムセットアップ実行中...\"\nbash \"$REPO_DIR/scripts/setup_server.sh\" || true\n\n# 2. ワールドデータ復元\necho \">> [2/7] ストレージからワールドデータ復元中...\"\npython3 \"$REPO_DIR/scripts/backup_restore.py\" restore || true\n\n# 3. LLM推論基盤の準備 (0円/L4 GPU最適化)\necho \">> [3/7] LLM推論エンジンの準備中...\"\nif [ -z \"$GEMINI_API_KEY\" ]; then\n    echo \"ローカルLLM (Ollama) を確認中...\"\n    if ! command -v ollama >/dev/null 2>&1; then\n        echo \"Ollama をインストール中 (数秒)...\"\n        curl -fsSL https://ollama.com/install.sh | sh || true\n    fi\n\n    # Ollamaサーバー起動\n    echo \"Ollama サーバーをバックグラウンド起動中...\"\n    pkill -f \"ollama serve\" || true\n    sleep 1\n    OLLAMA_HOST=0.0.0.0:8000 nohup ollama serve > /content/ollama.log 2>&1 &\n    sleep 3\n\n    echo \"日本語対応LLMモデル (qwen2.5:7b) をダウンロード中 (約4.5GB、1〜2分)...\"\n    ollama pull qwen2.5:7b || echo \"モデルロードを再試行します...\"\n\n    export LLM_API_URL=\"http://localhost:8000/v1/chat/completions\"\n    export LLM_MODEL=\"qwen2.5:7b\"\nelse\n    echo \"Google AI Studio (Gemini 1.5 Free Tier) を使用します。\"\nfi\n\n# 4. Minecraft サーバー起動 (Named Pipeによる安全なCUI入出力)\necho \">> [4/7] Minecraftサーバー (PaperMC) 起動中...\"\ncd \"$SERVER_DIR\"\n\n# 入力パイプ作成 (コマンド配信用)\nPIPE_PATH=\"$SERVER_DIR/server.pipe\"\nrm -f \"$PIPE_PATH\"\nmkfifo \"$PIPE_PATH\" 2>/dev/null || true\n\n# サーバー起動 (nohupバックグラウンド)\ntail -f \"$PIPE_PATH\" 2>/dev/null | java -Xms8G -Xmx12G -XX:+UseG1GC -jar server.jar nogui > \"$SERVER_DIR/logs/console.log\" 2>&1 &\nMC_PID=$!\necho \"$MC_PID\" > /content/mc_server.pid\n\n# サーバー起動完了待ち\necho \"サーバーの起動完了を待機中 (約30秒)...\"\nfor i in {1..30}; do\n    if grep -q \"Done\" \"$SERVER_DIR/logs/latest.log\" 2>/dev/null || grep -q \"Done\" \"$SERVER_DIR/logs/console.log\" 2>/dev/null; then\n        break\n    fi\n    sleep 2\ndone\necho \"✅ Minecraftサーバーが正常に起動しました！\"\n\n# 5. playit.gg トンネル起動\necho \">> [5/7] playit.gg トンネル接続中...\"\npkill -f \"playit\" || true\nnohup playit --stdout > /content/playit.log 2>&1 &\nsleep 5\n\necho \"\"\necho \"==================================================\"\necho \"★ Minecraft 接続先アドレスの確認 ★\"\necho \"playit.gg の接続ステータス:\"\ngrep -E \"claim|tunnel|connect|setup at|https://\" /content/playit.log 2>/dev/null || cat /content/playit.log | head -n 10\necho \"--------------------------------------------------\"\necho \"手元のJava版Minecraftクライアントから上記アドレスで接続してください。\"\necho \"==================================================\"\necho \"\"\n\n# 6. Bot依存関係インストール & AIPlayerクラスタ起動\necho \">> [6/7] AIPlayer群 (Gemini, ChatGPT, Claude, Grok) 起動中...\"\ncd \"$REPO_DIR/bot\"\nif [ ! -d \"node_modules\" ]; then\n    npm install --silent > /dev/null 2>&1 || true\nfi\npkill -f \"node index.js\" || true\nnohup node index.js > /content/bots.log 2>&1 &\n\n# 7. 画面視認 & 録画開始\necho \">> [7/7] Bot視点自動録画システム起動中...\"\nnohup bash \"$REPO_DIR/recorder/record_viewer.sh\" > /content/recorder.log 2>&1 &\n\necho \"\"\necho \"==================================================\"\necho \"    全システム起動完了！サバイバルをお楽しみください\"\necho \"==================================================\"\necho \"※ 終了時は、ノートブックの「シャットダウンセル」を実行してください。\"\necho \"※ 以下にサーバーのログ・動向をリアルタイム表示します...\"\necho \"\"\n\n# バックグラウンド定期バックアップ (10分ごと)\n(\n    while true; do\n        sleep 600\n        echo \"\"\n        echo \"[AutoBackup] 定期バックアップを実行中...\"\n        python3 \"$REPO_DIR/scripts/backup_restore.py\" backup || true\n    done\n) &\n\n# サーバーコンソールログをリアルタイム監視・出力\ntouch \"$SERVER_DIR/logs/latest.log\"\ntail -f \"$SERVER_DIR/logs/latest.log\"\n", "scripts/shutdown.sh": "#!/bin/bash\n# ==============================================================================\n# shutdown.sh - 安全な停止・バックアップ・Git自動プッシュ (堅牢化版)\n# ==============================================================================\n\nREPO_DIR=\"$(cd \"$(dirname \"${BASH_SOURCE[0]}\")/..\" && pwd)\"\nSERVER_DIR=\"/content/minecraft_server\"\nPIPE_PATH=\"$SERVER_DIR/server.pipe\"\n\necho \"==================================================\"\necho \"    Minecraft AI Colab Server - 安全なシャットダウン\"\necho \"==================================================\"\n\n# 1. 録画プロセスの終了\nif [ -f \"/content/recorder_pids.txt\" ]; then\n    echo \">> [1/5] 録画プロセスを停止中...\"\n    kill $(cat /content/recorder_pids.txt) 2>/dev/null || true\n    rm -f /content/recorder_pids.txt\nfi\npkill -f \"ffmpeg\" || true\npkill -f \"Xvfb\" || true\n\n# 2. AIPlayer ボットの停止\necho \">> [2/5] AIPlayer をログアウト中...\"\npkill -f \"node index.js\" || true\n\n# 3. Minecraft サーバーへの save-all & stop\necho \">> [3/5] Minecraftワールドを保存して停止中...\"\nif [ -p \"$PIPE_PATH\" ]; then\n    echo \"say サーバーを安全にシャットダウンします...\" > \"$PIPE_PATH\" 2>/dev/null || true\n    sleep 1\n    echo \"save-all\" > \"$PIPE_PATH\" 2>/dev/null || true\n    sleep 3\n    echo \"stop\" > \"$PIPE_PATH\" 2>/dev/null || true\n    sleep 8\nfi\npkill -f \"server.jar\" || true\nrm -f \"$PIPE_PATH\"\n\n# 4. playit.gg の停止\npkill -f \"playit\" || true\n\n# 5. Google Drive / XServer への最終バックアップ同期\necho \">> [4/5] 最終バックアップをストレージに同期中...\"\npython3 \"$REPO_DIR/scripts/backup_restore.py\" backup || true\n\n# 6. 行動日誌をリポジトリにコピーしてGitHubプッシュ\necho \">> [5/5] 行動日誌の保存 & GitHubプッシュ...\"\nif [ -f \"/content/diary.md\" ]; then\n    mkdir -p \"$REPO_DIR/logs\"\n    TIMESTAMP=$(date +\"%Y%m%d_%H%M%S\")\n    cp /content/diary.md \"$REPO_DIR/logs/diary_${TIMESTAMP}.md\"\n    cd \"$REPO_DIR\"\n    git add logs/\n    git commit -m \"chore: プレイ記録・行動日誌の保存 (${TIMESTAMP})\" || true\n    git push origin main || true\nfi\n\necho \"==================================================\"\necho \"    シャットダウンが完了しました。お疲れ様でした！\"\necho \"==================================================\"\n", "bot/package.json": "{\n  \"name\": \"colab-minecraft-ai-bots\",\n  \"version\": \"1.0.0\",\n  \"description\": \"Multi-Agent AI Players (Gemini, ChatGPT, Claude, Grok) for Minecraft Colab\",\n  \"main\": \"index.js\",\n  \"scripts\": {\n    \"start\": \"node index.js\"\n  },\n  \"dependencies\": {\n    \"axios\": \"^1.7.7\",\n    \"mineflayer\": \"^4.20.1\",\n    \"mineflayer-auto-eat\": \"^3.3.6\",\n    \"mineflayer-collectblock\": \"^1.4.1\",\n    \"mineflayer-pathfinder\": \"^2.4.5\",\n    \"mineflayer-pvp\": \"^1.3.2\",\n    \"prismarine-viewer\": \"^1.27.0\",\n    \"vec3\": \"^0.1.10\"\n  }\n}\n", "bot/persona.js": "/**\n * persona.js - AIPlayerのキャラクター・性格・役割の定義\n */\n\nmodule.exports = {\n  Gemini: {\n    name: \"Gemini\",\n    role: \"リーダー・建築・全体進行\",\n    personality: \"明るく協調的で親しみやすい。サバイバル全体の計画を立てたり、拠点の建築や装飾、住環境の整備が得意。\",\n    firstGreeting: \"こんにちは！Geminiです！みんなで一緒に最高の拠点を作りましょう！何から始めますか？\",\n    systemPrompt: `あなたはMinecraft内の仲間プレイヤー「Gemini」です。\nプレイヤーと一緒にサバイバル生活を送っています。\n【特徴】\n- 明るく協調的で、みんなのリーダー役です。\n- 建築や拠点作り、サバイバル全体の計画立案が得意です。\n- プレイヤーの発言に親切に応え、日本語で短く自然にチャットします（Minecraftのチャット欄なので1〜2文程度が好ましい）。\n- プレイヤーの入力に誤字・脱字やあいまいな表現があっても、前後の文脈から優しく意図を汲み取ってください。`\n  },\n\n  ChatGPT: {\n    name: \"ChatGPT\",\n    role: \"採掘・クラフト・素材管理\",\n    personality: \"論理的で効率的、丁寧な口調。鉱石の採掘や道具のクラフト、チェストの整理整頓が得意。\",\n    firstGreeting: \"お疲れ様です、ChatGPTです。必要なツールや鉱石の採掘、クラフトはお任せください。\",\n    systemPrompt: `あなたはMinecraft内の仲間プレイヤー「ChatGPT」です。\nプレイヤーと一緒にサバイバル生活を送っています。\n【特徴】\n- 丁寧で論理的、効率重視のサポーターです。\n- 採掘（木材・石・鉄・ダイヤ）、クラフト、素材管理・チェスト整理を担当します。\n- Minecraftのチャットに適した短く簡潔な日本語で返答します（1〜2文程度）。\n- プレイヤーの意図を正確に推論し、必要な素材や作業を提案します。`\n  },\n\n  Claude: {\n    name: \"Claude\",\n    role: \"護衛・戦闘・安全食料管理\",\n    personality: \"誠実で落ち着いており、安全第一。周囲のモンスターへの警戒、戦闘、食料の確保や農業が得意。\",\n    firstGreeting: \"はじめまして、Claudeです。周囲の安全と食料の確保は私にお任せを。無理のないよう進みましょう。\",\n    systemPrompt: `あなたはMinecraft内の仲間プレイヤー「Claude」です。\nプレイヤーと一緒にサバイバル生活を送っています。\n【特徴】\n- 誠実で落ち着いた性格、安全と思慮深さを重視します。\n- プレイヤーの周囲の安全確認、夜間のモンスター迎撃、食料（農業や狩り）の確保を担当します。\n- 簡潔で思いやりのある日本語で返答します（1〜2文程度）。\n- プレイヤーが危険な状態（夜間・低HP・モンスター接近）なら即座に警告します。`\n  },\n\n  Grok: {\n    name: \"Grok\",\n    role: \"探索・冒険・ムードメーカー\",\n    personality: \"率直で好奇心旺盛、少し皮肉やユーモアを交える。未知のバイオームや洞窟の探索、レアアイテム探しが得意。\",\n    firstGreeting: \"よっ、Grokだ！じっとしてる暇はないぜ、未知のバイオームやダンジョンをガンガン探索しよう！\",\n    systemPrompt: `あなたはMinecraft内の仲間プレイヤー「Grok」です。\nプレイヤーと一緒にサバイバル生活を送っています。\n【特徴】\n- 好奇心旺盛で率直、ユーモアと冒険心にあふれる性格です。\n- 遠征、バイオーム探索、洞窟探検、珍しい構造物の発見を担当します。\n- フランクで生き生きとした日本語で返答します（1〜2文程度）。\n- 退屈を嫌い、新しい冒険や面白い行動を提案します。`\n  }\n};\n", "bot/brain.js": "/**\n * brain.js - LLM推論基盤・文脈管理・行動コマンド抽出\n * ローカル推論(vLLM/Ollama) 0円 または Google AI Studio Free Tier\n */\n\nconst axios = require('axios');\n\nclass AIBrain {\n  constructor(options = {}) {\n    // ローカルvLLM/Ollama (OpenAI互換API) または Google AI Studio\n    this.apiUrl = process.env.LLM_API_URL || 'http://localhost:8000/v1/chat/completions';\n    this.geminiApiKey = process.env.GEMINI_API_KEY || '';\n    this.modelName = process.env.LLM_MODEL || 'default';\n    this.chatHistory = []; // 直近のチャット・行動履歴\n    this.maxHistory = 15;\n  }\n\n  addHistory(sender, text) {\n    this.chatHistory.push({\n      sender,\n      text,\n      time: new Date().toLocaleTimeString('ja-JP')\n    });\n    if (this.chatHistory.length > this.maxHistory) {\n      this.chatHistory.shift();\n    }\n  }\n\n  getFormattedHistory() {\n    return this.chatHistory.map(h => `[${h.time}] ${h.sender}: ${h.text}`).join('\\n');\n  }\n\n  /**\n   * ユーザーのチャットに対してLLMで思考し、返答と行動コマンドを決定する\n   */\n  async think(persona, userMessage, playerPos, botPos, nearbyEntities = []) {\n    this.addHistory('User', userMessage);\n\n    const historyStr = this.getFormattedHistory();\n    const contextPrompt = `\nあなたはMinecraftのAI仲間「${persona.name}」(${persona.role})です。\nプレイヤーの発言には誤字・脱字や口語、あいまいな表現が含まれている場合があります。\n前後の文脈や現在の状況からプレイヤーの真意を推測し、柔軟に対応してください。\n\n【現在の状況】\n- あなたの位置: X:${Math.round(botPos.x)}, Y:${Math.round(botPos.y)}, Z:${Math.round(botPos.z)}\n- プレイヤーの位置: X:${Math.round(playerPos.x)}, Y:${Math.round(playerPos.y)}, Z:${Math.round(playerPos.z)}\n- 周囲の生物/モブ: ${nearbyEntities.join(', ') || '特になし'}\n\n【直近の会話履歴】\n${historyStr}\n\n【あなたの行動選択肢 (action)】\n- \"chat\": ただ会話する、返事をする\n- \"follow\": プレイヤーについていく\n- \"collect\": 木や石などの素材を集める\n- \"attack\": 近くの敵対モブを攻撃する・護衛する\n- \"explore\": 付近を探索・見回りする\n- \"stop\": 現在の行動を中止して待機する\n\n必ず以下のJSON形式でのみ出力してください。余計な解説は不要です。\n{\n  \"reply\": \"Minecraftのゲーム内チャットで発言する日本語（1〜2文）\",\n  \"action\": \"chat | follow | collect | attack | explore | stop\",\n  \"target\": \"アクションの対象（木材、ゾンビ、探索方向など。なければnull）\"\n}\n`;\n\n    // 1. Google AI Studio (Gemini Free Tier) が指定されている場合\n    if (this.geminiApiKey) {\n      return await this.callGeminiApi(persona, contextPrompt, userMessage);\n    }\n\n    // 2. ローカルL4推論 (vLLM / Ollama: コスト0円)\n    return await this.callLocalLlm(persona, contextPrompt, userMessage);\n  }\n\n  async callLocalLlm(persona, systemPrompt, userMessage) {\n    try {\n      const response = await axios.post(\n        this.apiUrl,\n        {\n          model: this.modelName,\n          messages: [\n            { role: 'system', content: persona.systemPrompt + '\\n' + systemPrompt },\n            { role: 'user', content: userMessage }\n          ],\n          temperature: 0.7,\n          max_tokens: 300\n        },\n        { timeout: 15000 }\n      );\n\n      const content = response.data.choices[0].message.content;\n      return this.parseJsonResponse(content, persona);\n    } catch (err) {\n      console.error(`[${persona.name}] LLM推論エラー:`, err.message);\n      // フォールバック返答\n      return {\n        reply: `了解です！まかせてください！`,\n        action: 'follow',\n        target: null\n      };\n    }\n  }\n\n  async callGeminiApi(persona, systemPrompt, userMessage) {\n    try {\n      const url = `https://generativelanguage.googleapis.com/v1beta/models/gemini-1.5-flash:generateContent?key=${this.geminiApiKey}`;\n      const response = await axios.post(\n        url,\n        {\n          contents: [\n            {\n              role: 'user',\n              parts: [{ text: persona.systemPrompt + '\\n' + systemPrompt + '\\nユーザー発言: ' + userMessage }]\n            }\n          ],\n          generationConfig: {\n            temperature: 0.7,\n            maxOutputTokens: 300\n          }\n        },\n        { timeout: 15000 }\n      );\n\n      const content = response.data.candidates[0].content.parts[0].text;\n      return this.parseJsonResponse(content, persona);\n    } catch (err) {\n      console.error(`[${persona.name}] Gemini APIエラー:`, err.message);\n      return {\n        reply: `はい、了解しました！`,\n        action: 'follow',\n        target: null\n      };\n    }\n  }\n\n  parseJsonResponse(raw, persona) {\n    try {\n      // JSONブロック抽出\n      const jsonMatch = raw.match(/\\{[\\s\\S]*\\}/);\n      if (jsonMatch) {\n        const parsed = JSON.parse(jsonMatch[0]);\n        if (parsed.reply) return parsed;\n      }\n    } catch (e) {\n      // パース失敗時は生の文字列から適当にトリミング\n    }\n    // 文字列そのままの場合\n    return {\n      reply: raw.replace(/[\"{}\\\\]/g, '').trim().slice(0, 80),\n      action: 'chat',\n      target: null\n    };\n  }\n}\n\nmodule.exports = AIBrain;\n", "bot/index.js": "/**\n * index.js - Multi-AIPlayer Manager for Minecraft Colab\n * 4体のAIPlayer (Gemini, ChatGPT, Claude, Grok) を同時制御\n */\n\nconst mineflayer = require('mineflayer');\nconst { pathfinder, Movements, goals } = require('mineflayer-pathfinder');\nconst pvp = require('mineflayer-pvp').plugin;\nconst autoEat = require('mineflayer-auto-eat').plugin;\nconst mineflayerViewer = require('prismarine-viewer').mineflayer;\nconst fs = require('fs');\nconst path = require('path');\n\nconst personas = require('./persona');\nconst AIBrain = require('./brain');\n\nconst SERVER_HOST = process.env.MC_HOST || 'localhost';\nconst SERVER_PORT = parseInt(process.env.MC_PORT || '25565', 10);\nconst DIARY_PATH = process.env.DIARY_PATH || '/content/diary.md';\n\nconst brain = new AIBrain();\nconst activeBots = [];\n\n// 行動日誌の初期化\nfunction appendDiary(text) {\n  const line = `[${new Date().toISOString()}] ${text}\\n`;\n  try {\n    fs.appendFileSync(DIARY_PATH, line, 'utf8');\n  } catch (e) {\n    // ignore\n  }\n}\n\nfunction createBot(personaKey, index) {\n  const persona = personas[personaKey];\n  console.log(`[Manager] Bot作成中: ${persona.name} (${persona.role})`);\n\n  const bot = mineflayer.createBot({\n    host: SERVER_HOST,\n    port: SERVER_PORT,\n    username: persona.name,\n    version: false // 自動検出\n  });\n\n  bot.loadPlugin(pathfinder);\n  bot.loadPlugin(pvp);\n  bot.loadPlugin(autoEat);\n\n  bot.once('spawn', () => {\n    console.log(`[${persona.name}] サーバーにスポーンしました！`);\n    appendDiary(`${persona.name} がサーバーにスポーンしました。`);\n\n    const mcData = require('minecraft-data')(bot.version);\n    const defaultMove = new Movements(bot, mcData);\n    bot.pathfinder.setMovements(defaultMove);\n\n    // 1体目（Gemini）のみPrismarine-viewerの3D視点描画を有効化（ポート3000）\n    if (index === 0) {\n      try {\n        mineflayerViewer(bot, { port: 3000, firstPerson: true });\n        console.log(`[Viewer] Prismarine-Viewer がポート 3000 で起動しました (Bot: ${persona.name})`);\n      } catch (err) {\n        console.error('[Viewer] 起動エラー:', err.message);\n      }\n    }\n\n    // 初回挨拶\n    setTimeout(() => {\n      bot.chat(persona.firstGreeting);\n      brain.addHistory(persona.name, persona.firstGreeting);\n    }, (index + 1) * 2000);\n  });\n\n  // チャット受信イベント\n  bot.on('chat', async (username, message) => {\n    // 自身の発言、または他のBotの発言なら無視\n    if (username === bot.username || Object.keys(personas).some(k => personas[k].name === username)) {\n      return;\n    }\n\n    console.log(`[Chat] <${username}> ${message}`);\n    const lowerMsg = message.toLowerCase();\n    const isTargeted =\n      message.includes(persona.name) ||\n      (personaKey === 'Gemini' && !Object.keys(personas).some(k => message.includes(personas[k].name)));\n\n    // 自分宛ての指示、またはリーダー(Gemini)宛ての汎用発言の場合に対応\n    if (isTargeted || message.includes('みんな') || message.includes('全員')) {\n      const player = bot.players[username]?.entity;\n      const playerPos = player ? player.position : bot.entity.position;\n      const botPos = bot.entity.position;\n\n      // 周囲のモブを取得\n      const nearbyMobs = Object.values(bot.entities)\n        .filter(e => e.type === 'mob' && e.position.distanceTo(botPos) < 16)\n        .map(e => e.name);\n\n      bot.chat(`思案中...`);\n      const thought = await brain.think(persona, message, playerPos, botPos, nearbyMobs);\n\n      // 返答\n      if (thought.reply) {\n        bot.chat(thought.reply);\n        brain.addHistory(persona.name, thought.reply);\n        appendDiary(`<${persona.name}> [返答]: ${thought.reply} (アクション: ${thought.action})`);\n      }\n\n      // 行動実行\n      executeAction(bot, persona, thought, player);\n    }\n  });\n\n  // 自動食事\n  bot.on('health', () => {\n    if (bot.food < 15) {\n      bot.autoEat.eat().catch(() => {});\n    }\n  });\n\n  bot.on('death', () => {\n    console.log(`[${persona.name}] 倒れました。リスポーンします...`);\n    appendDiary(`${persona.name} が倒れました。`);\n  });\n\n  bot.on('error', (err) => {\n    console.error(`[${persona.name}] エラー:`, err.message);\n  });\n\n  return bot;\n}\n\n// 行動実行ロジック\nfunction executeAction(bot, persona, thought, playerEntity) {\n  const { action, target } = thought;\n  if (!bot.pathfinder) return;\n\n  switch (action) {\n    case 'follow':\n      if (playerEntity) {\n        console.log(`[${persona.name}] プレイヤーを追従開始`);\n        const goal = new goals.GoalFollow(playerEntity, 2);\n        bot.pathfinder.setGoal(goal, true);\n      }\n      break;\n\n    case 'stop':\n      console.log(`[${persona.name}] 停止・待機`);\n      bot.pathfinder.setGoal(null);\n      if (bot.pvp) bot.pvp.stop();\n      break;\n\n    case 'attack':\n      const targetMob = bot.nearestEntity(e => e.type === 'mob' && e.kind === 'Hostile');\n      if (targetMob && bot.pvp) {\n        console.log(`[${persona.name}] モブを攻撃: ${targetMob.name}`);\n        bot.pvp.attack(targetMob);\n      } else {\n        bot.chat('周囲に敵は見当たりません！');\n      }\n      break;\n\n    case 'collect':\n    case 'explore':\n      // 付近の安全なランダム座標へ移動\n      const rx = bot.entity.position.x + (Math.random() * 20 - 10);\n      const rz = bot.entity.position.z + (Math.random() * 20 - 10);\n      const ry = bot.entity.position.y;\n      bot.pathfinder.setGoal(new goals.GoalNear(rx, ry, rz, 1));\n      break;\n\n    default:\n      break;\n  }\n}\n\n// 4体のBotを順次起動\nconsole.log('=== Minecraft AIPlayer クラスタ起動中 ===');\nconst personaKeys = ['Gemini', 'ChatGPT', 'Claude', 'Grok'];\n\npersonaKeys.forEach((key, idx) => {\n  setTimeout(() => {\n    const b = createBot(key, idx);\n    activeBots.push(b);\n  }, idx * 4000); // 接続が重ならないよう4秒間隔で参加\n});\n", "recorder/record_viewer.sh": "#!/bin/bash\n# ==============================================================================\n# record_viewer.sh - Prismarine-Viewer & ffmpeg によるBot視点自動録画\n# ==============================================================================\n\nRECORDINGS_DIR=\"/content/recordings\"\nmkdir -p \"$RECORDINGS_DIR\"\n\nTIMESTAMP=$(date +\"%Y%m%d_%H%M%S\")\nOUTPUT_FILE=\"${RECORDINGS_DIR}/mc_play_${TIMESTAMP}.mp4\"\n\necho \"=== [1/3] 仮想ディスプレイ (Xvfb :99) 起動 ===\"\nexport DISPLAY=:99\nXvfb :99 -screen 0 1280x720x24 > /dev/null 2>&1 &\nXVFB_PID=$!\nsleep 2\n\necho \"=== [2/3] Prismarine-Viewer (Bot視点) のブラウザ描画開始 ===\"\n# Chromium / Google ChromeでPrismarine-viewerのURLを開く\nif which chromium-browser > /dev/null; then\n    BROWSER=\"chromium-browser\"\nelif which chromium > /dev/null; then\n    BROWSER=\"chromium\"\nelif which google-chrome > /dev/null; then\n    BROWSER=\"google-chrome\"\nelse\n    apt-get update -qq && apt-get install -y -qq chromium-browser > /dev/null 2>&1\n    BROWSER=\"chromium-browser\"\nfi\n\n# Viewerが起動するまで待機\nuntil curl -s http://localhost:3000 > /dev/null; do\n    echo \"Prismarine-Viewer の起動待機中...\"\n    sleep 3\ndone\n\n$BROWSER --no-sandbox --disable-gpu --window-size=1280,720 --start-fullscreen \"http://localhost:3000\" > /dev/null 2>&1 &\nBROWSER_PID=$!\nsleep 5\n\necho \"=== [3/3] ffmpeg による画面収録開始 (出力: $OUTPUT_FILE) ===\"\n# 20fps, 1280x720, H.264\nffmpeg -y -f x11grab -draw_mouse 0 -video_size 1280x720 -framerate 20 -i :99.0 \\\n    -c:v libx264 -preset ultrafast -pix_fmt yuv420p \"$OUTPUT_FILE\" > /dev/null 2>&1 &\nFFMPEG_PID=$!\n\necho \"録画がバックグラウンドで開始されました (PID: $FFMPEG_PID)\"\n\n# プロセスIDを保存\necho \"$XVFB_PID $BROWSER_PID $FFMPEG_PID\" > /content/recorder_pids.txt\n"}
for file_path, content in EMBEDDED_FILES.items():
    full_path = os.path.join(REPO_PATH, file_path)
    os.makedirs(os.path.dirname(full_path), exist_ok=True)
    with open(full_path, 'w', encoding='utf-8') as f:
        f.write(content)

print('✅ スクリプト一式を /content/ColabTool に展開しました。')

# 3. 実行権限の付与と起動スクリプトの実行
%cd /content/ColabTool
!chmod +x scripts/*.sh scripts/*.py recorder/*.sh
!bash scripts/start_all.sh


---
## 💬 ステップ 2: サーバーコマンド送信（任意・必要な時のみ）
自分に管理者権限（OP）を付与したい時や、天候・ゲームルールの変更などに使えます。

In [ ]:
#@title サーバーコンソールへのコマンド送信 (例: op <自分のMCユーザー名>)
command = "op your_mc_username" #@param {type:"string"}
pipe_path = "/content/minecraft_server/server.pipe"
if os.path.exists(pipe_path):
    with open(pipe_path, "w") as f:
        f.write(f"{command}\n")
    print(f"送信完了: {command}")
    !tail -n 10 /content/minecraft_server/logs/latest.log
else:
    print("サーバーパイプが見つかりません。サーバーが起動しているか確認してください。")


---
## 🛑 ステップ 3: 【安全なシャットダウン】
ゲームを終えるときは、**このセルを実行** してください。  
ワールドデータをGoogle Drive/XServerへ確実にバックアップし、動画を保存、プレイ行動日誌を保存して安全に終了します。

In [ ]:
#@title 🛑 サーバー安全シャットダウン & バックアップ
import os
if os.path.exists('/content/ColabTool'):
    %cd /content/ColabTool
    !bash scripts/shutdown.sh
else:
    print('サーバーが起動されていません。')
